# Canada University Enrollments — Cleaning & Category Mapping

Source: Statistics Canada, Table 37-10-0011-01  
"Postsecondary enrolments, by field of study, registration status, program type, credential type and gender"  
Release date: 2025-11-20, covering academic years 2016/2017 to 2023/2024.

**Coverage notes:**
- Institution type: Total (universities + colleges)
- Credential type: Total (all credential levels, not bachelor's only)
- Registration status: Total (full-time + part-time)
- Gender: Total
- All counts are randomly rounded to a multiple of 3 by Statistics Canada

**CIP → Category key mapping:**

| CIP | CIP name | Category key | Category name | Notes |
|-----|----------|-------------|---------------|-------|
| [1] | Education | 7 | Education | Direct 1:1 |
| [2] | Visual and performing arts, and communications technologies | 10 | Creative Arts | Direct 1:1 |
| [3]+[4] | Humanities + Social and behavioural sciences and law | 9 | Society & Culture | Combined |
| [5] | Business, management and public administration | 8 | Management & Commerce | Direct 1:1 |
| [6] | Physical and life sciences and technologies | 1 | Natural & Physical Science | Direct 1:1 |
| [7] | Mathematics, computer and information sciences | 2 | Information Technology | Direct 1:1 |
| [8] | Architecture, engineering and related technologies | 3 | Engineering & Related Tech | Includes Architecture; see note below |
| [9] | Agriculture, natural resources and conservation | 5 | Environment & Related | Direct 1:1 |
| [10] | Health and related fields | 6 | Health | Direct 1:1 |
| [12] | Other field of study | 11 | Others | Direct 1:1 |

**Architecture (catkey=4):** CIP [8] combines Architecture with Engineering and cannot be disaggregated 
from this table. Canada is therefore excluded as a control for the Architecture regression. 
CIP [8] is assigned to Engineering (catkey=3) as that is the dominant component.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

raw_path = ROOT / 'data' / 'raw' / 'CANADA University Enrollments 2016-2024.csv'
print('Project root:', ROOT)
print('Raw file exists:', raw_path.exists())

Project root: C:\Users\neddp\ECC3479-Project-JRGS
Raw file exists: True


In [2]:
# Read raw file — row 14 (0-indexed) is the header; row 15 is a "Number" subheader to skip
skip_rows = list(range(14)) + [15]
raw = pd.read_csv(raw_path, skiprows=skip_rows, header=0)
# Rename first column to field_of_study
raw = raw.rename(columns={raw.columns[0]: 'field_of_study'})
# Strip whitespace from column names
raw.columns = [str(c).strip() for c in raw.columns]
print('Raw shape:', raw.shape)
display(raw)

Raw shape: (39, 9)


,field_of_study,2016 / 2017,2017 / 2018,2018 / 2019,2019 / 2020,2020 / 2021,2021 / 2022,2022 / 2023,2023 / 2024
0,"Total, field of study","2,076,105","2,116,545","2,156,445","2,185,539","2,173,071","2,197,533","2,213,391","2,342,547"
1,Personal improvement and leisure [0],"20,232","21,264","22,179","21,885","14,529","15,096","16,026","18,810"
2,Education [1],"91,881","91,011","92,556","95,454","98,469","101,751","100,767","102,846"
3,"Visual and performing arts, and communications...","81,333","82,305","83,007","84,060","82,272","82,449","82,866","86,709"
4,Humanities [3],"278,829","267,900","268,554","259,038","255,510","239,490","232,173","239,910"
5,Social and behavioural sciences and law [4],"287,424","288,942","292,158","297,840","307,584","308,196","302,469","311,307"
6,"Business, management and public administration...","374,307","388,281","397,557","409,371","413,934","419,496","431,559","476,952"
7,Physical and life sciences and technologies [6],"168,195","171,432","173,505","177,312","182,028","186,159","186,033","193,563"
8,"Mathematics, computer and information sciences...","82,794","92,928","103,533","115,413","123,261","131,688","144,210","164,601"
9,"Architecture, engineering and related technolo...","221,493","227,688","229,971","235,707","228,705","231,648","238,263","250,725"


In [3]:
# Rename field-of-study column and drop rows we won't use
raw.columns = ['field_of_study'] + [c.strip() for c in raw.columns[1:]]

# Drop aggregate and unmappable rows
EXCLUDE = [
    'Total, field of study',
    'Personal improvement and leisure  [0]',
    'Personal, protective and transportation services  [11]',
    'Unclassified, field of study 9',
]
raw = raw[~raw['field_of_study'].isin(EXCLUDE)].copy()
print('Rows after exclusion:', len(raw))
display(raw[['field_of_study']])

Rows after exclusion: 35


,field_of_study
2,Education [1]
3,"Visual and performing arts, and communications..."
4,Humanities [3]
5,Social and behavioural sciences and law [4]
6,"Business, management and public administration..."
7,Physical and life sciences and technologies [6]
8,"Mathematics, computer and information sciences..."
9,"Architecture, engineering and related technolo..."
10,"Agriculture, natural resources and conservatio..."
11,Health and related fields [10]


In [4]:
# Strip commas from enrollment counts and convert to numeric
year_cols = [c for c in raw.columns if '/' in str(c)]
print('Year columns found:', year_cols)

for col in year_cols:
    raw[col] = pd.to_numeric(raw[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

# Map academic year label to start year integer: '2016 / 2017' → 2016
year_map = {col: int(col.strip().split('/')[0].strip()) for col in year_cols}
print('Year mapping:', year_map)

Year columns found: ['2016 / 2017', '2017 / 2018', '2018 / 2019', '2019 / 2020', '2020 / 2021', '2021 / 2022', '2022 / 2023', '2023 / 2024']
Year mapping: {'2016 / 2017': 2016, '2017 / 2018': 2017, '2018 / 2019': 2018, '2019 / 2020': 2019, '2020 / 2021': 2020, '2021 / 2022': 2021, '2022 / 2023': 2022, '2023 / 2024': 2023}


In [5]:
# CIP → category_key mapping (matched by CIP bracket code for robustness)
# Society & Culture (9) sums CIP [3] Humanities + CIP [4] Social & behavioural sciences and law
# Architecture (catkey=4) NOT mappable — CIP [8] is assigned to Engineering (3) only

import re

def cip_code(field_name):
    """Extract the CIP bracket code, e.g. '[8]' from the field name."""
    m = re.search(r'\[(\d+)\]', str(field_name))
    return int(m.group(1)) if m else None

CIP_CATKEY = {
    1:  7,   # Education → Education
    2:  10,  # Visual and performing arts → Creative Arts
    3:  9,   # Humanities → Society & Culture (part 1)
    4:  9,   # Social & behavioural sciences and law → Society & Culture (part 2)
    5:  8,   # Business, management and public administration → Management & Commerce
    6:  1,   # Physical and life sciences → Natural & Physical Science
    7:  2,   # Mathematics, computer and information sciences → Information Technology
    8:  3,   # Architecture, engineering and related tech → Engineering (incl. Architecture)
    9:  5,   # Agriculture, natural resources and conservation → Environment & Related
    10: 6,   # Health and related fields → Health
    12: 11,  # Other field of study → Others
}

raw['cip_code'] = raw['field_of_study'].map(cip_code)
raw['category_key'] = raw['cip_code'].map(CIP_CATKEY)

unmapped = raw[raw['category_key'].isna() & raw['cip_code'].notna()]
excluded = raw[raw['cip_code'].isna()]
print('Excluded (no CIP code — aggregate/unclassified rows):')
display(excluded[['field_of_study']])
print('Unmapped CIP codes (should be empty):')
display(unmapped[['field_of_study', 'cip_code']])

raw = raw[raw['category_key'].notna()].copy()
print(f'Rows remaining for mapping: {len(raw)}')
display(raw[['field_of_study', 'cip_code', 'category_key'] + year_cols])

Excluded (no CIP code — aggregate/unclassified rows):


,field_of_study
15,NaN
16,NaN
17,Table Corrections:
18,Date
19,12/02/2019
20,13/01/2023
21,NaN
22,NaN
23,NaN
24,Footnotes:


Unmapped CIP codes (should be empty):


,field_of_study,cip_code


Rows remaining for mapping: 11


,field_of_study,cip_code,category_key,2016 / 2017,2017 / 2018,2018 / 2019,2019 / 2020,2020 / 2021,2021 / 2022,2022 / 2023,2023 / 2024
2,Education [1],1.0,7.0,91881.0,91011.0,92556.0,95454.0,98469.0,101751.0,100767.0,102846.0
3,"Visual and performing arts, and communications...",2.0,10.0,81333.0,82305.0,83007.0,84060.0,82272.0,82449.0,82866.0,86709.0
4,Humanities [3],3.0,9.0,278829.0,267900.0,268554.0,259038.0,255510.0,239490.0,232173.0,239910.0
5,Social and behavioural sciences and law [4],4.0,9.0,287424.0,288942.0,292158.0,297840.0,307584.0,308196.0,302469.0,311307.0
6,"Business, management and public administration...",5.0,8.0,374307.0,388281.0,397557.0,409371.0,413934.0,419496.0,431559.0,476952.0
7,Physical and life sciences and technologies [6],6.0,1.0,168195.0,171432.0,173505.0,177312.0,182028.0,186159.0,186033.0,193563.0
8,"Mathematics, computer and information sciences...",7.0,2.0,82794.0,92928.0,103533.0,115413.0,123261.0,131688.0,144210.0,164601.0
9,"Architecture, engineering and related technolo...",8.0,3.0,221493.0,227688.0,229971.0,235707.0,228705.0,231648.0,238263.0,250725.0
10,"Agriculture, natural resources and conservatio...",9.0,5.0,37269.0,38778.0,39372.0,40290.0,39933.0,39921.0,39669.0,40347.0
11,Health and related fields [10],10.0,6.0,255558.0,260607.0,262647.0,265227.0,268164.0,275124.0,277008.0,287493.0


In [6]:
# Melt to long format and aggregate (Society & Culture sums CIP [3]+[4])
melted = raw[['category_key'] + year_cols].melt(
    id_vars=['category_key'], var_name='academic_year', value_name='total_enrollments'
)
melted['year'] = melted['academic_year'].map(year_map)
melted = melted.drop(columns=['academic_year'])

# Sum within category_key × year (handles Society & Culture combination)
canada = melted.groupby(['category_key', 'year'], as_index=False)['total_enrollments'].sum()
canada['category_key'] = canada['category_key'].astype(int)
canada['year'] = canada['year'].astype(int)
canada['total_enrollments'] = canada['total_enrollments'].astype(int)

# Add readable field_of_study label
KEY_LABELS = {
    1: 'Natural & Physical Science',
    2: 'Information Technology',
    3: 'Engineering & Related Tech',
    5: 'Environment & Related',
    6: 'Health',
    7: 'Education',
    8: 'Management & Commerce',
    9: 'Society & Culture',
    10: 'Creative Arts',
    11: 'Others',
}
canada['field_of_study'] = canada['category_key'].map(KEY_LABELS)
canada = canada[['field_of_study', 'category_key', 'year', 'total_enrollments']].sort_values(['category_key', 'year']).reset_index(drop=True)

print(f'Clean Canada data: {len(canada)} rows, {canada["category_key"].nunique()} disciplines, years {canada["year"].min()}–{canada["year"].max()}')
print('Note: Canada data ends at 2023 (Statistics Canada 2023/2024 academic year = start year 2023).')
print('Architecture (catkey=4) excluded — CIP [8] combines Engineering + Architecture indistinguishably.')
display(canada)

Clean Canada data: 80 rows, 10 disciplines, years 2016–2023
Note: Canada data ends at 2023 (Statistics Canada 2023/2024 academic year = start year 2023).
Architecture (catkey=4) excluded — CIP [8] combines Engineering + Architecture indistinguishably.


,field_of_study,category_key,year,total_enrollments
0,Natural & Physical Science,1,2016,168195
1,Natural & Physical Science,1,2017,171432
2,Natural & Physical Science,1,2018,173505
3,Natural & Physical Science,1,2019,177312
4,Natural & Physical Science,1,2020,182028
...,...,...,...,...
75,Others,11,2019,53445
76,Others,11,2020,50718
77,Others,11,2021,49545
78,Others,11,2022,47619


In [7]:
# Pivot to wide format for visual check
wide = canada.pivot_table(index='field_of_study', columns='year', values='total_enrollments')
wide.columns.name = None
print('Canada enrollments by discipline and year:')
display(wide)

Canada enrollments by discipline and year:


,2016,2017,2018,2019,2020,2021,2022,2023
field_of_study,,,,,,,,
Creative Arts,81333.0,82305.0,83007.0,84060.0,82272.0,82449.0,82866.0,86709.0
Education,91881.0,91011.0,92556.0,95454.0,98469.0,101751.0,100767.0,102846.0
Engineering & Related Tech,221493.0,227688.0,229971.0,235707.0,228705.0,231648.0,238263.0,250725.0
Environment & Related,37269.0,38778.0,39372.0,40290.0,39933.0,39921.0,39669.0,40347.0
Health,255558.0,260607.0,262647.0,265227.0,268164.0,275124.0,277008.0,287493.0
Information Technology,82794.0,92928.0,103533.0,115413.0,123261.0,131688.0,144210.0,164601.0
Management & Commerce,374307.0,388281.0,397557.0,409371.0,413934.0,419496.0,431559.0,476952.0
Natural & Physical Science,168195.0,171432.0,173505.0,177312.0,182028.0,186159.0,186033.0,193563.0
Others,52896.0,54204.0,55302.0,53445.0,50718.0,49545.0,47619.0,50142.0


In [8]:
# Export clean file
out_path = ROOT / 'data' / 'clean' / 'canada_enrollments_2016_2024.csv'
canada.to_csv(out_path, index=False)
print(f'Exported: {out_path}')
print(f'Rows: {len(canada)}, Columns: {list(canada.columns)}')

Exported: C:\Users\neddp\ECC3479-Project-JRGS\data\clean\canada_enrollments_2016_2024.csv


Rows: 80, Columns: ['field_of_study', 'category_key', 'year', 'total_enrollments']
